# bridge

> IPython namespace bridge

One indirection with three jobs: pick the namespace via an injected **client** —
`IPythonClient` (the live IPython `user_ns`) by default, or `NsClient(dict)` for headless
`serve()` and tests — expose the frontend-agnostic verbs (`snapshot`/`view`/`expand`/`grid`/
`set_value`) over it, and hook `post_run_cell` so in-kernel frontends refresh after every
execution. Point paar at a different namespace by constructing `Bridge` with a different
client — no module globals involved, so it's trivial to test with a real dict.

In [ ]:
#| default_exp bridge

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.bridge import Bridge, NsClient, IPythonClient, on_change
from fastcore.test import test_eq

# NsClient: a real dict-backed namespace source -- no IPython, no mocks
b = Bridge(NsClient({'a': 1, '_ih': 2}, hidden={'_ih'}))
test_eq([r.name for r in b.snapshot()], ['a'])          # hidden names drop out of the default view

# IPythonClient + on_change against a real, running IPython shell
import IPython, paar.bridge as B
from IPython.core.interactiveshell import InteractiveShell
try: ip = InteractiveShell.instance()          # a real shell in plain IPython
except Exception: ip = IPython.get_ipython()   # under execnb, reuse the already-running shell
B.get_ipython = lambda: ip
ip.user_ns['q_probe'] = 42
test_eq('q_probe' in [r.name for r in Bridge().snapshot()], True)   # IPythonClient reads the live shell
hits = []; test_eq(on_change(lambda: hits.append(1)), True)
ip.events.callbacks['post_run_cell'][-1](None)                      # invoke the registered cb (real EventManager)
test_eq(hits, [1])

# no kernel -> empty snapshot, and on_change is a no-op
B.get_ipython = lambda: None
test_eq(Bridge().snapshot(), []); test_eq(on_change(lambda: None), False)
B.get_ipython = IPython.get_ipython                                 # restore the real accessor

In [ ]:
# expand, name/type filters, and set_value over a real dict namespace
b = Bridge(NsClient({'d': {'x': 1, 'y': 2}, 'alpha': 1, 'beta': 'x'}))
kids = b.expand(('d',))
test_eq([k.name for k in kids], ["'x'", "'y'"]); test_eq(kids[0].value, '1')
test_eq([r.name for r in b.snapshot(name='a')], ['alpha', 'beta'])   # name substring filter
test_eq([r.name for r in b.snapshot(typ='int')], ['alpha'])          # type filter

nb = NsClient({'x': 1})
test_eq(Bridge(nb).set_value(('x',), '7'), None); test_eq(nb.ns()['x'], 7)   # write lands in the dict

In [ ]:
import numpy as np
b = Bridge(NsClient({'arr': np.arange(6).reshape(2, 3)}))
p = b.grid(('arr',)); test_eq((p['nrows'], p['ncols']), (2, 3))

In [ ]:
import math
view = Bridge(NsClient({'myvar': 5, 'mymod': math})).view('standard')
labels = [l for l, _ in view]
test_eq(None in labels and 'Modules' in labels, True)
test_eq([v.name for v in dict(view)[None]], ['myvar'])

In [ ]:
#| export
from paar.snapshot import snapshot, expand, grid_page, profile_view
from paar.runtime import set_value
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

class IPythonClient:
    "Default namespace source: the live IPython kernel (empty when not in IPython)."
    def ns(self):     ip = get_ipython(); return ip.user_ns if ip else {}
    def hidden(self): ip = get_ipython(); return frozenset(getattr(ip, 'user_ns_hidden', ())) if ip else frozenset()
    def set_value(self, accessor, expr): return set_value(accessor, expr)

class NsClient:
    "A plain-dict namespace source for headless serve() and real (non-mock) tests."
    def __init__(self, ns, hidden=frozenset()): self.ns_, self.hidden_ = ns, frozenset(hidden)
    def ns(self):     return self.ns_
    def hidden(self): return self.hidden_
    def set_value(self, accessor, expr): return set_value(accessor, expr, ns=self.ns_)

class Bridge:
    "Frontend-agnostic access to a namespace via an injected client (defaults to the live IPython kernel)."
    def __init__(self, client=None): self.client = client or IPythonClient()
    def snapshot(self, name=None, typ=None): return snapshot(self.client.ns(), self.client.hidden(), name, typ)
    def view(self, profile='standard'): return profile_view(self.client.ns(), self.client.hidden(), profile)
    def expand(self, accessor, offset=0): return expand(self.client.ns(), accessor, offset)
    def grid(self, accessor, roff=0, coff=0, rows=50, cols=50): return grid_page(self.client.ns(), accessor, roff, coff, rows, cols)
    def set_value(self, accessor, expr): return self.client.set_value(accessor, expr)

def on_change(cb):
    "Register cb() to fire after each cell execution; returns False outside IPython."
    ip = get_ipython()
    if not ip: return False
    ip.events.register('post_run_cell', lambda res: cb())
    return True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()